# 머신러닝 기반텍스트 분류 
1. 데이터 준비 : 로딩, 입력데이터, 출력데이터, 학습/테스트 데이터, 특징추출
2. 학습&평가 
3. 배포 준비 

## 1. 데이터 준비


In [1]:
import pandas as pd
datafile = './data/Korean_movie_reviews_2016.csv'
data_df = pd.read_csv(datafile)
data_df.head()

,review,label
0,부산 행 때문 너무 기대하고 봤,0
1,한국 좀비 영화 어색하지 않게 만들어졌 놀랍,1
2,조금 전 보고 왔 지루하다 언제 끝나 이 생각 드,0
3,평 밥 끼 먹자 돈 니 내고 미친 놈 정신사 좀 알 싶어 그래 밥 먹다 먹던 숟가락...,1
4,점수 대가 과 이 엑소 팬 어중간 점수 줄리 없겠 클레멘타인 이후 최고 평점 조작 ...,0


In [2]:
data_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 165384 entries, 0 to 165383
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   review  165384 non-null  object
 1   label   165384 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 2.5+ MB


In [3]:
# 입력 데이터, 정답 데이터 추출 
review_list = list(data_df.review) # 입력 데이터
label_list = list(data_df.label) # 출력 데이터
len(review_list), len(label_list)

(165384, 165384)

In [4]:
# 학습 데이터, 테스트 데이터 분리
from sklearn.model_selection import train_test_split

train_X, test_X, train_y, test_y = train_test_split(review_list, label_list, test_size=0.1)
len(train_X), len(test_X), len(train_y), len(test_y)

(148845, 16539, 148845, 16539)

In [5]:
# 한국어 토크나이저 정의
from konlpy.tag import Okt
def korean_tokenizer(text):
    my_tags = ['Noun', 'Adjective', 'Verb']
    my_stopwords = []
    tokenizer = Okt().pos
    return [word for word, tag in tokenizer(text) if tag in my_tags and word not in my_stopwords]

In [65]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=1000) #tokenizer=korean_tokenizer를 삭제해도 실행은 된다.
vectorizer.fit(train_X)

TfidfVectorizer(max_features=1000)

In [66]:
len(vectorizer.get_feature_names_out()), vectorizer.get_feature_names_out()[:10]

(1000,
 array(['가고', '가는', '가면', '가볍', '가서', '가슴', '가장', '가족', '가지', '가치'],
       dtype=object))

In [69]:
# 학습 데이터 특징 추출
train_X_fv = vectorizer.transform(train_X)

In [70]:
# 테스트 데이터 특징 추출
test_X_fv = vectorizer.transform(test_X)

In [71]:
print(train_X_fv)

  (np.int32(0), np.int32(524))	0.400841207855061
  (np.int32(0), np.int32(678))	0.6302223872406707
  (np.int32(0), np.int32(726))	0.38876851227832115
  (np.int32(0), np.int32(884))	0.40100591116369705
  (np.int32(0), np.int32(927))	0.36083150053541574
  (np.int32(1), np.int32(382))	0.5411973889582475
  (np.int32(1), np.int32(884))	0.342225949590938
  (np.int32(1), np.int32(886))	0.5190635111167584
  (np.int32(1), np.int32(983))	0.566180057082991
  (np.int32(2), np.int32(108))	0.5233511886106538
  (np.int32(2), np.int32(155))	0.3492966129510867
  (np.int32(2), np.int32(370))	0.3869729709815875
  (np.int32(2), np.int32(431))	0.36017486185563463
  (np.int32(2), np.int32(646))	0.5697555600234827
  (np.int32(3), np.int32(20))	0.4082331518987786
  (np.int32(3), np.int32(303))	0.30912464622235064
  (np.int32(3), np.int32(415))	0.27446562771352145
  (np.int32(3), np.int32(529))	0.4732842114790217
  (np.int32(3), np.int32(639))	0.26817633815877445
  (np.int32(3), np.int32(912))	0.47024996621155

In [73]:
# 정답 데이터를 ndarray 변환 np.array(sequence)
import numpy as np
train_y = np.array(train_y)
test_y = np.array(test_y)
print(train_y[:10])

[1 1 0 1 1 0 0 0 0 0]


# 2. 머신러닝 - 모델 학습 
1. 의사결정트리, Decision Tree (DT)
2. 랜덤포레스트, Rendom forest (RF)

In [14]:
# 모델별 정확도를 dataframe으로 저장하여 비교
import pandas as pd
score_df = pd.DataFrame(columns=['train', 'test'])
score_df

,train,test


In [23]:
def get_scores(model, train_X, train_y, test_X, test_y):
    train_score = model.score(train_X, train_y) * 100
    test_score =  model.score(test_X, test_y) * 100
    return train_score, test_score

### 1.

In [24]:
from sklearn.tree import DecisionTreeClassifier

dtc = DecisionTreeClassifier()
dtc.fit(train_X_fv, train_y)

DecisionTreeClassifier()

In [19]:
train_score, test_score = get_scores(dtc, train_X_fv, train_y, test_X_fv, test_y)
print(train_score, test_score)

0.9811145822835836 0.8006530019952839


In [20]:
score_df.loc['DecisionTree'] = [train_score, test_score]
score_df

,train,test
DecisionTree,0.981115,0.800653


### 2. Random Forest

In [74]:
from sklearn.ensemble import RandomForestClassifier
rf =RandomForestClassifier(n_jobs=-1)
rf.fit(train_X_fv, train_y)

RandomForestClassifier(n_jobs=-1)

In [75]:
train_score, test_score = get_scores(rf, train_X_fv, train_y, test_X_fv, test_y)
print(train_score, test_score)
score_df.loc['RandomForest'] = [train_score, test_score]
score_df

95.82989015418725 83.27589334300744


,train,test
DecisionTree,0.981115,0.800653
RandomForest,95.829890,83.275893
Naivebayes,85.320300,85.198621
NaiveBayes,85.320300,85.198621
LogisticRegression,85.320300,85.198621
SVM,86.204441,85.797207


### 3.나이브 베이즈 분류

In [76]:
from sklearn.naive_bayes import MultinomialNB

mnb = MultinomialNB()
mnb.fit(train_X_fv, train_y)

MultinomialNB()

In [77]:
train_score, test_score = get_scores(mnb, train_X_fv, train_y, test_X_fv, test_y)
print(train_score, test_score)
score_df.loc['NaiveBayes'] = [train_score, test_score]
score_df

84.16675064664584 84.17679424390833


,train,test
DecisionTree,0.981115,0.800653
RandomForest,95.829890,83.275893
Naivebayes,85.320300,85.198621
NaiveBayes,84.166751,84.176794
LogisticRegression,85.320300,85.198621
SVM,86.204441,85.797207


### 4. 로지스틱 회귀 분석, logistic regression

In [78]:
from sklearn.linear_model import LogisticRegression
lr = LogisticRegression(solver='liblinear')
lr.fit(train_X_fv, train_y)

LogisticRegression(solver='liblinear')

In [79]:
train_score, test_score = get_scores(mnb, train_X_fv, train_y, test_X_fv, test_y)
print(train_score, test_score)
score_df.loc['LogisticRegression'] = [train_score, test_score]
score_df

84.16675064664584 84.17679424390833


,train,test
DecisionTree,0.981115,0.800653
RandomForest,95.829890,83.275893
Naivebayes,85.320300,85.198621
NaiveBayes,84.166751,84.176794
LogisticRegression,84.166751,84.176794
SVM,86.204441,85.797207


### 5. 서포트 벡터 머신, SVM

In [80]:
from sklearn.svm import LinearSVC
svm = LinearSVC()
svm.fit(train_X_fv, train_y)

LinearSVC()

In [81]:
train_score, test_score = get_scores(svm, train_X_fv, train_y, test_X_fv, test_y)
print(train_score, test_score)
score_df.loc['SVM'] = [train_score, test_score]
score_df

84.88629110819981 84.55166575971946


,train,test
DecisionTree,0.981115,0.800653
RandomForest,95.829890,83.275893
Naivebayes,85.320300,85.198621
NaiveBayes,84.166751,84.176794
LogisticRegression,84.166751,84.176794
SVM,84.886291,84.551666


In [84]:
score_df.sort_values(by='test', ascending=False)

,train,test
Naivebayes,85.320300,85.198621
SVM,84.886291,84.551666
NaiveBayes,84.166751,84.176794
LogisticRegression,84.166751,84.176794
RandomForest,95.829890,83.275893
DecisionTree,0.981115,0.800653


### 6. 배포 준비
-기능 구현

-모델 저장

In [ ]:
#review = '영화가 너무 재미있다'
review = '이게 영화나? 나도 만들겠다'
# review = '재미가 없다'

def analyze_sentiment(review):
    # 전처리 및 특징 벡터 추출
    
    review_fv = vectorizer.transform([review])
    #print(review_fv)

    result = rf.predict(review_fv)
    #print(result)

    show = '긍정' if result[0] >= 0.5 else '부정'
    return show

show = analyze_sentiment(review)
print(f'{review} -> {show}')

이게 영화나? 나도 만들겠다 -> 부정


In [62]:
review = [
    '영화가 너무 재미있다',
    '이게 영화냐? 나도 만들겠다',
    '개꿀잼',
    '핵노잼'
]

for review in review:
    print(f'{review} -> {analyze_sentiment(review)}')

영화가 너무 재미있다 -> 긍정
이게 영화냐? 나도 만들겠다 -> 부정
개꿀잼 -> 긍정
핵노잼 -> 부정


In [64]:
import joblib

vectorizer_file = './model/sa_movie_vectorizer.pkl'
joblib.dump(vectorizer, vectorizer_file)

model_file = './model/sa_movie_model.pkl'
joblib.dump(rf, model_file)

['./model/sa_movie_model.pkl']